##Refencences
https://www.mygreatlearning.com/blog/what-is-feature-engineering/

https://aioconquer.aivietnam.edu.vn/posts/du-an-machine-learning-du-doan-diem-thi-cua-hoc-sinh


## 4. Feature Engineering

Ở giai đoạn, này bạn cần chuẩn bị, lọc chọn lại các đặc trưng cho dữ liệu.

Hãy tưởng tượng hai học sinh chuẩn bị đi thi:
- Một phải ôn bài với mớ tài liệu hỗn độn.
- Một được ôn bài với những tài liệu đã được chắt lọc.

Bạn nghĩ học sinh nào đạt được kết quả cao hơn? Kết quả phần lớn sẽ là học sinh 2.

Trong Machine Learning, bước Feature Engineering (kỹ thuật tạo đặc trưng) chính là quá trình chuyển các tài liệu hỗn độn (dữ liệu thô) thành tài liệu hữu ích(feature) để cải thiện kết quả của các học sinh (mô hình ML).

### 4.1 Feature Creation

Trong bài toán dự đoán PM2.5, các yếu tố của môi trường có ảnh hưởng cao đến kết quả thực tế. Bên cạnh đó đặc điểm của **Time Series là phụ thuộc theo thời gian**, các yếu tố trong quá khứ vẫn tạo ra ảnh hưởng tới hiện tại. Từ hai đặc điểm trên nhóm tạo ra các đặc trưng tiêu biểu như sau:

#### *Lag Features*:

Được tạo bằng cách lấy các giá trị pm25_avg trong quá khứ làm feature cho dự đoán hiện tại.

|index|pm25\_avg|pm25\_avg\_lag\_1|pm25\_avg\_lag\_2|
|---|---|---|---|
|0|29\.13999938964844|NaN|NaN|
|1|29\.15000009536743|29\.13999938964844|NaN|
|2|31\.78333314259847|29\.15000009536743|29\.13999938964844|
|3|30\.95000012715657|31\.78333314259847|29\.15000009536743|
|4|30\.21666669845581|30\.95000012715657|31\.78333314259847|

```python
#Chỉ số pm25_avg n giờ trước
pm25_df['pm25_avg_lag1'] = pm25_df['pm25_avg'].shift(1)
pm25_df['pm25_avg_lag6'] = pm25_df['pm25_avg'].shift(6)
pm25_df['pm25_avg_lag24'] = pm25_df['pm25_avg'].shift(24)

#Các chỉ số môi trường từ n giờ trước
pm25_df['temperature_2m_lag1'] = pm25_df['temperature_2m'].shift(1)
```

#### *Window Features*:

Tổng hợp thông tin từ các giờ trước bằng các phương pháp thống kê.

```python
#Tạo Rolling feature cho pm25
pm25_df['pm25_avg_rolling_mean_3h'] = pm25_df['pm25_avg'].rolling(window=3).mean()
pm25_df['pm25_avg_rolling_mean_6h'] = pm25_df['pm25_avg'].rolling(window=6).mean()
pm25_df['pm25_avg_rolling_mean_12h'] = pm25_df['pm25_avg'].rolling(window=12).mean()
```

#### *Domain Features:*

Các đặc trưng được tạo dựa trên kiến thức chuyên ngành:

ventilation_coeff (Hệ số thông thoáng)
$$
Ventilation = Wind\_Speed \times Boundary\_Layer\_Height
$$
- **Ý nghĩa:** Đại diện cho khả năng khuếch tán và pha loãng bụi trong khí quyển.
- Giá trị thấp cho thấy không khí bị “nén” lại, làm ô nhiễm dễ tích tụ hơn.

```python
#Tạo ventilation_coeff
pm25_df['ventilation_coeff'] = pm25_df['wind_speed_10m'] * pm25_df['boundary_layer_height']
```
dew_point
$$
Dew\ Point = Temperature - \frac{100 - Relative\ Humidity}{5}
$$
- **Ý nghĩa:** Là chỉ số liên quan đến độ ẩm không khí, hỗ trợ mô tả trạng thái khí quyển thuận lợi cho ô nhiễm.

```python
#Tạo biến dew_point
pm25_df['dew_point'] = pm25_df['temperature_2m'] - (100 - pm25_df['relative_humidity_2m']) / 5
```
> Các đặc trưng được tạo thêm khi Feature Engineering không phải là ngẫu nhiên mà cần được tạo dựa trên hiểu biết về bài toán và đặc điểm của dữ liệu.

### 4.2 Feature Transformation và Encoding

Feature Transformation là quá trình thay đổi giá trị hoặc phân phối của feature để làm cho dữ liệu phù hợp hơn với mô hình. Kỹ thuật chuẩn hóa dữ liệu cho các biến kiểu số của nhóm là:

#### *Standard Scaler (Chuẩn hóa Z-score)*

$$
z = \frac{x - \mu}{\sigma}
$$

Standard Scaler là kỹ thuật chuẩn hóa dữ liệu bằng cách đưa mỗi đặc trưng về phân phối có: Mean = 0 và Standard Deviation = 1

Kỹ thuật được sử dụng cho bài toán dự đoán pm2.5 nhằm:
- Đưa các biến về cùng thang
- Tránh biến có giá trị lớn (relative_humidity_2m ~ 60) “lấn át” các biến có giá trị nhỏ (precipitation ~ 0.1).

#### *Encoding*

![One Hot Encoding](https://raw.githubusercontent.com/tienphamMLE/Image/main/one%20hot%20encoding.png)

*ảnh tạo bởi AI One-Hot Encoding (OHE): Tạo các cột nhị phân (0/1) cho từng giá trị danh mục (phù hợp với các biến phân loại có số lượng giá trị không quá lớn).*

One Hot Encoding được sử dụng trong bài toán để giải quyết hai vấn đề chính:
- Các mô hình được tính toán trên số nên không hiểu dữ liệu dạng category
- Tránh được mô hình hiểu sai khi encoding dạng: Cloudy (2) > Rainy (1) > Sunny (0).


```python
# Tạo preprocessor: mã hóa chuỗi (OneHot) + chuẩn hóa số (Scaler)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ])
# Biến đổi dữ liệu
X_train_proc = preprocessor.fit_transform(X_train)
X_valid_proc = preprocessor.transform(X_valid)
X_test_proc  = preprocessor.transform(X_test)
```
### 4.3 Feature Selection

>Theo qui tắc "Rác vào, rác ra", input không tốt -> model đoán tệ.

Feature Engineering không chỉ là tạo thêm các feature, mà còn là loại bỏ những feature tệ, gây ảnh hưởng xấu tới mô hình. Sau đây là các kỹ thuật selection mà nhóm sử dụng.

#### *Filter Methods (Phương pháp lọc)*

Phương pháp lọc (Filter Methods) là cách chọn đặc trưng dựa trên các tiêu chí thống kê, mà không cần huấn luyện mô hình. Giúp lựa chọn các đặc trưng trước khi huấn luyện mô hình.

![Pearson_Correlation](https://raw.githubusercontent.com/tienphamMLE/Image/main/coleration.png)

```python
correlation_matrix = pm25_df_processed.select_dtypes(include=np.number).corr()

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(18, 14))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation Matrix of Numerical Features (Processed Data)')
plt.show()
```

#### *Embedded Methods (Phương pháp tích hợp):*

Các mô hình cây (như Decision Tree, Random Forest, XGBoost) có khả năng tự chọn đặc trưng quan trọng trong quá trình huấn luyện. Nhờ cơ chế này, mô hình cây có thể cung cấp chỉ số feature importance, hỗ trợ việc phân tích và lựa chọn lại tập đặc trưng một cách hiệu quả.

![fe_importance](https://raw.githubusercontent.com/tienphamMLE/Image/main/fe%20im.png)

```python
cat_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)
```

#### *SHAP (SHapley Additive exPlanations)*

SHAP (SHapley Additive exPlanations) là một phương pháp trong học máy dùng để giải thích dự đoán của mô hình bằng cách phân bổ một cách công bằng mức đóng góp của từng đặc trưng vào kết quả dự đoán cuối cùng.

![fe_importance](https://raw.githubusercontent.com/tienphamMLE/Image/main/shap.png)








